# PhC Waveguide Dataset — Data Visualisation
### `phc_out_profile.h5`  ·  Design 9 walkthrough

---

## 1 · What is a Photonic Crystal (PhC) Waveguide?

A **photonic crystal waveguide** is a dielectric structure whose refractive-index profile is modulated *periodically* in space. In this dataset the waveguide is fabricated in **Si₃N₄ on SiO₂** — a standard telecom platform.

The core idea is that the periodic corrugation creates a photonic *stop-band* (bandgap): wavelengths that satisfy the Bragg condition are strongly reflected, while others propagate with low loss.

**How the geometry is built:**  
- A silicon nitride ridge waveguide has its *lateral width* modulated sinusoidally along the propagation direction (x-axis).  
- The modulation repeats with a fixed period of ≈ 0.49 µm.  
- The width profile along one period is parameterised by 100 sample points (`y_width_arrays`), giving a rich design space for machine-learning.  
- The full 2-D cross-section at each point along the waveguide is rasterised into a binary **image matrix** (2000 × 490 pixels), which is used as the geometry input to a convolutional neural network.

The plot below shows the image matrix for **design 9** — white pixels are Si₃N₄, black pixels are cladding (SiO₂ / air).

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

H5_PATH   = 'phc_out_profile.h5'
DESIGN_ID = 9
DNAME     = f'design_{DESIGN_ID:04d}'

# ── Load shared wavelength axis and design-9 data ──────────────────────────
with h5py.File(H5_PATH, 'r') as f:
    wl        = f['wavelengths_nm'][:]
    grp       = f[DNAME]
    y_width   = grp['y_width_arrays'][:]
    img_mat   = grp['image_matrices'][:]        # (2000, 490)
    idx_img   = grp['index_images'][:]          # (40, 20)
    s11_n10   = grp['S_n_10/s11_power'][:]
    s11_n20   = grp['S_n_20/s11_power'][:]
    s12_n10   = grp['S_n_10/s12_power'][:]
    s12_n20   = grp['S_n_20/s12_power'][:]
    err_n10   = float(grp['S_n_10/simulation_error'][()])
    err_n20   = float(grp['S_n_20/simulation_error'][()])
    loss_spec = grp['scattering_loss_per_period_dB'][:]
    avg_loss9 = float(grp['average_scattering_loss_dB'][()])

print(f'Wavelength range : {wl[0]:.0f} – {wl[-1]:.0f} nm  ({len(wl)} points)')
print(f'image_matrices   : {img_mat.shape}')
print(f'index_images     : {idx_img.shape}')

# ── Plot: binary image matrix ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))
ax.imshow(img_mat.T, origin='lower', cmap='gray', aspect='auto')
ax.set_xlabel('x  (pixels along propagation)')
ax.set_ylabel('y  (transverse pixels)')
ax.set_title(f'Design {DESIGN_ID} — Binary geometry mask  (image_matrix, 2000×490)\n'
             f'White = Si₃N₄ core,  Black = SiO₂/air cladding')
plt.tight_layout()
plt.show()

---

## 2 · What the image above is — and what it is NOT

The image matrix above is a **binary geometry mask** generated from the CAD design.  It is a *design-space representation*, not a simulation snapshot.

In a real FDTD simulation, the electromagnetic fields fill the entire domain including evanescent tails in the cladding, substrate leakage, and scattered radiation.  The solver discretises Maxwell's equations on a Yee mesh with sub-wavelength grid spacing.  At each time step the fields are advanced until they decay to a preset auto-shutoff threshold (typically −60 dB). Source and monitor positions are placed in the waveguide mode basis, so only guided-mode power is recorded.  What the FDTD solver actually "sees" is far richer than the cartoon mask above.

The next plot shows the **refractive-index map** (`index_images`, 40×20 pixels) extracted from the actual FDTD simulation at λ ≈ 1550 nm.  This is the epsilon profile the solver uses on its computational mesh — the waveguide core, substrate, and any mode discontinuities at the corrugations are all visible here.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(idx_img.T, origin='lower', cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, label='Refractive index  n')
ax.set_xlabel('x  (mesh units along propagation)')
ax.set_ylabel('y  (transverse mesh units)')
ax.set_title(f'Design {DESIGN_ID} — FDTD refractive-index map\n'
             f'(index_images, 40×20,  extracted at λ ≈ 1550 nm)')
plt.tight_layout()
plt.show()
print(f'n range in this design: {idx_img.min():.3f} – {idx_img.max():.3f}')

---

## 3 · What is a "period" and why simulate two different lengths?

The PhC waveguide has a **spatial period** of ≈ 0.49 µm.  Simulating N periods at once gives N × 0.49 µm of total waveguide length.

We simulate two lengths:
- **N = 10** → 4.9 µm total  
- **N = 20** → 9.8 µm total  

The reason we need **two lengths** is the **cutback method** — a standard technique to separate bulk propagation loss from *facet coupling loss* (power lost at the input/output port interfaces, which is the same regardless of length).

$$\text{loss per period (dB)} = \frac{IL_{n=20}(\lambda) - IL_{n=10}(\lambda)}{20 - 10}$$

Because the facet losses cancel in the subtraction, what remains is purely the per-period scattering loss of the PhC corrugations themselves.

---

## 4 · Reflectance spectrum (S11)

`S11` is the power fraction reflected back to the input port.  It is the primary spectral signature of a PhC:  
- In the **stop-band** (Bragg condition), S11 → 1 (almost all light reflected).  
- In the **pass-band**, S11 ≈ 0 (light propagates through).  

The stop-band position and width depend on the period geometry.  Comparing S11 for N=10 and N=20 at the same design shows how the bandgap deepens with more periods.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wl, s11_n10, label='S11  N=10 periods', color='steelblue', lw=2)
ax.plot(wl, s11_n20, label='S11  N=20 periods', color='firebrick', lw=2, linestyle='--')
ax.set_xlabel('Wavelength  (nm)')
ax.set_ylabel('Reflectance  |S11|²')
ax.set_title(f'Design {DESIGN_ID} — Reflectance spectrum\n'
             f'Peak wavelength tells you where the Bragg stop-band is centred')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# annotate the peak
peak_wl = wl[np.argmax(s11_n20)]
peak_val = s11_n20.max()
ax.annotate(f'Stop-band peak\n{peak_wl:.0f} nm\nR = {peak_val:.2f}',
            xy=(peak_wl, peak_val), xytext=(peak_wl + 8, peak_val - 0.15),
            arrowprops=dict(arrowstyle='->', color='firebrick'),
            color='firebrick', fontsize=9)
plt.tight_layout()
plt.show()
print(f'N=10 peak reflectance : {s11_n10.max():.3f}  at {wl[np.argmax(s11_n10)]:.1f} nm')
print(f'N=20 peak reflectance : {s11_n20.max():.3f}  at {wl[np.argmax(s11_n20)]:.1f} nm')

---

## 5 · Scattering loss per period

The reflectance tells us *where* light is blocked, but not *how much light is truly lost* to radiation and scattering.  For a neural network training target we care about the **propagation loss** — how much power is absorbed or scattered out of the guided mode **per unit cell**.

Using the cutback formula with forward-direction insertion loss:
$$\text{scattering\_loss\_per\_period\_dB}(\lambda) = \frac{IL_{fwd,\,n20}(\lambda) - IL_{fwd,\,n10}(\lambda)}{10}$$

Wavelengths where this quantity is ≤ 0 are **excluded** (they correspond to Fabry-Pérot interference fringes, not physical gain).  `average_scattering_loss_dB` is the mean over the remaining positive wavelengths.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wl, loss_spec, color='darkorange', lw=2, label='scattering loss/period (fwd, +ve only)')
ax.axhline(avg_loss9, color='darkred', lw=1.5, linestyle='--',
           label=f'average = {avg_loss9:.3f} dB/period')
ax.set_xlabel('Wavelength  (nm)')
ax.set_ylabel('Scattering loss  (dB / period)')
ax.set_title(f'Design {DESIGN_ID} — Scattering loss spectrum\n'
             f'NaN gaps = Fabry-Pérot interference fringes excluded from average')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

n_valid = int(np.sum(np.isfinite(loss_spec)))
print(f'Valid (positive) wavelength points : {n_valid} / {len(wl)}')
print(f'Average scattering loss            : {avg_loss9:.4f} dB/period')

---

## 6 · Average scattering loss across all designs

The plot below shows `average_scattering_loss_dB` for every design in the dataset.

**NaN values** (shown as gaps) can arise from two causes:
1. **Genuinely low-loss design** — all wavelength points have `fwd_loss ≤ 0` because the device scatters so little that interference fringes dominate the entire spectrum.
2. **Simulation incomplete or failed** — the FDTD run did not finish or produced unphysical S-parameters.

In either case the design may still be valid for training on transmission/reflectance targets; it simply has no reliable loss label.

In [ ]:
# ── Load average loss and simulation error for ALL designs ─────────────────
with h5py.File(H5_PATH, 'r') as f:
    design_keys = sorted(k for k in f.keys() if k.startswith('design_'))
    n_designs   = len(design_keys)
    indices     = np.arange(n_designs)
    avg_loss_all = np.full(n_designs, np.nan, dtype=np.float32)
    err_n10_all  = np.full(n_designs, np.nan, dtype=np.float32)
    err_n20_all  = np.full(n_designs, np.nan, dtype=np.float32)
    for j, key in enumerate(design_keys):
        grp = f[key]
        avg_loss_all[j] = grp['average_scattering_loss_dB'][()]
        if 'S_n_10/simulation_error' in grp:
            err_n10_all[j] = grp['S_n_10/simulation_error'][()]
        if 'S_n_20/simulation_error' in grp:
            err_n20_all[j] = grp['S_n_20/simulation_error'][()]

n_nan  = int(np.sum(np.isnan(avg_loss_all)))
n_valid = n_designs - n_nan
print(f'Total designs : {n_designs}')
print(f'With valid average loss : {n_valid}   ({100*n_valid/n_designs:.1f}%)')
print(f'NaN (low-loss or failed): {n_nan}')

# ── Plot: average loss vs design index ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.scatter(indices[np.isfinite(avg_loss_all)],
           avg_loss_all[np.isfinite(avg_loss_all)],
           s=4, color='steelblue', alpha=0.6, label='valid loss estimate')
nan_idx = indices[np.isnan(avg_loss_all)]
if len(nan_idx):
    ax.scatter(nan_idx, np.zeros(len(nan_idx)),
               s=8, color='tomato', alpha=0.5, marker='x',
               label=f'NaN  (n={len(nan_idx)}, low-loss or sim error)')
ax.axhline(np.nanmean(avg_loss_all), color='navy', lw=1.2, linestyle='--',
           label=f'dataset mean = {np.nanmean(avg_loss_all):.3f} dB/period')
ax.set_xlabel('Design index')
ax.set_ylabel('Average scattering loss  (dB / period)')
ax.set_title('Average scattering loss per period — all designs\n'
             'Red crosses at y=0 mark designs with no reliable loss label (NaN)')
ax.legend(markerscale=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 7 · Simulation reliability — reciprocity error per design

The **simulation error** is defined as:

$$\text{simulation\_error}_i = \text{mean}_\lambda \left| S12_i(\lambda) - S21_i(\lambda) \right|$$

By Lorentz reciprocity, S12 = S21 exactly for any passive, linear waveguide.  In FDTD they differ only due to numerical artefacts (mesh asymmetry, mode-overlap approximation, finite auto-shutoff).  A small reciprocity error means the simulation has converged well.

The plot below shows this error across all designs for both N=10 and N=20 runs.  The dataset-level mean gives an overall confidence estimate for the S-parameter labels.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.scatter(indices[np.isfinite(err_n10_all)],
           err_n10_all[np.isfinite(err_n10_all)],
           s=4, color='steelblue', alpha=0.5, label='N=10')
ax.scatter(indices[np.isfinite(err_n20_all)],
           err_n20_all[np.isfinite(err_n20_all)],
           s=4, color='firebrick', alpha=0.5, label='N=20')

mean10 = np.nanmean(err_n10_all)
mean20 = np.nanmean(err_n20_all)
ax.axhline(mean10, color='steelblue', lw=1.2, linestyle='--',
           label=f'N=10 mean = {mean10:.4f}')
ax.axhline(mean20, color='firebrick', lw=1.2, linestyle='--',
           label=f'N=20 mean = {mean20:.4f}')

ax.set_xlabel('Design index')
ax.set_ylabel('Simulation error  mean|S12 − S21|')
ax.set_title('FDTD reciprocity error per design — both period counts\n'
             'Lower is better; mean values indicate overall dataset quality')
ax.legend(markerscale=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'N=10  mean±std simulation error : {mean10:.5f} ± {np.nanstd(err_n10_all):.5f}')
print(f'N=20  mean±std simulation error : {mean20:.5f} ± {np.nanstd(err_n20_all):.5f}')
print()
print('Interpretation: values below ~0.01 indicate well-converged FDTD runs.')
print('Outliers (high error) may warrant re-simulation or down-weighting in training.')